## Iris Flower Classification using SVM and Gradio
**In this notebook, we will demonstrate how to:**

- Load and explore the Iris dataset
- Preprocess the data using feature scaling
- Train a Support Vector Machine (SVM) classifier
- Evaluate the model performance
- Create a Gradio web interface for interactive prediction

This is a classic and complete example that shows how to go from raw data to a deployable machine learning model, using one of the most well-known datasets in machine learning.

In [1]:
# Import necessary libraries
import gradio as gr  # For creating the web interface
from sklearn import datasets  # To load built-in datasets like Iris
from sklearn.model_selection import train_test_split  # For splitting data into train/test sets
from sklearn.preprocessing import StandardScaler  # For feature scaling (normalization)
from sklearn.svm import SVC  # SVM classifier
from sklearn.metrics import classification_report  # For evaluation metrics

# Load the Iris dataset (150 samples, 4 features, 3 classes)
iris = datasets.load_iris()
X = iris.data  # Feature matrix: sepal length, sepal width, petal length, petal width
y = iris.target  # Target labels: 0 = setosa, 1 = versicolor, 2 = virginica
target_names = iris.target_names  # Class label names

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Preprocess the data: standardize features to have mean 0 and std 1
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # Fit and transform on training data
X_test = scaler.transform(X_test)  # Only transform test data using the same scaler

# Create and train an SVM classifier with a linear kernel
svm_model = SVC(kernel='linear', probability=True)  # probability=True allows probabilistic outputs
svm_model.fit(X_train, y_train)  # Train the model

# Evaluate the model on the test set
predictions = svm_model.predict(X_test)  # Predict class labels for the test set

# Print classification metrics like precision, recall, and F1-score
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=target_names))


Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.89      0.94         9
   virginica       0.92      1.00      0.96        11

    accuracy                           0.97        30
   macro avg       0.97      0.96      0.97        30
weighted avg       0.97      0.97      0.97        30



---
## Why Gradio?
Gradio allows us to create simple, shareable web interfaces for machine learning models. In this notebook, we’ll build a small app where users can input the four flower measurements and receive a prediction of the flower species — no coding needed for the user!

This makes it easier to:

- Demonstrate model behavior
- Allow non-technical users to interact with ML models
- Package your model for presentations or assignments

In [3]:
# Define the Gradio interface function
def predict_species(sepal_length, sepal_width, petal_length, petal_width):
    # Format the input features into a 2D array and scale them using the same scaler from training
    input_features = scaler.transform([[sepal_length, sepal_width, petal_length, petal_width]])
    
    # Get the predicted class probabilities from the trained SVM model
    probabilities = svm_model.predict_proba(input_features).flatten()
    
    # Find the index of the highest probability (i.e., the predicted class)
    prediction_index = probabilities.argmax()
    
    # Get the name of the predicted species
    predicted_species = target_names[prediction_index]
    
    # Format the predicted probability as a percentage (for better user understanding)
    probability_percentage = f"{probabilities[prediction_index] * 100:.2f}%"
    
    # Return a formatted string like: "setosa (98.23%)"
    return f"{predicted_species} ({probability_percentage})"

# Create the Gradio interface using the new Blocks API (allows flexible layout)
with gr.Blocks() as app:
    # Add a title to the app
    gr.Markdown("# Iris Species Predictor")

    # Arrange the input components side by side in a row
    with gr.Row():
        sepal_length = gr.Number(label="Sepal Length (cm)")
        sepal_width = gr.Number(label="Sepal Width (cm)")
        petal_length = gr.Number(label="Petal Length (cm)")
        petal_width = gr.Number(label="Petal Width (cm)")

    # Add a button to trigger the prediction
    predict_button = gr.Button("Predict Species")
    
    # Add a textbox to display the result
    species_output = gr.Textbox(label="Predicted Species")

    # Define what happens when the button is clicked:
    # it calls the `predict_species` function using the values from the inputs
    predict_button.click(
        predict_species,
        inputs=[sepal_length, sepal_width, petal_length, petal_width],
        outputs=[species_output]
    )

# Launch the Gradio app locally (a browser window will open with the UI)
app.launch()


* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.
